# 02 — Treino da CNN-A (Baseline)

Arquitetura simples (3 blocos Conv+Pool), **sem regularização e sem aumento de dados**. Objetivo: estabelecer o piso de desempenho e evidenciar o overfitting que a CNN-B vai atacar.

> Recomendado rodar no **Google Colab com GPU** (Runtime → Change runtime type → T4 GPU). Treino ~5 min.

In [ ]:
# >>> EXECUTE ESTA CELULA APENAS NO GOOGLE COLAB <<<
# Clona o repositorio e instala as dependencias. Em ambiente local, pule.
# !git clone https://github.com/SEU_USUARIO/gs-cnn-eurosat.git
# %cd gs-cnn-eurosat
# !pip install -q -r requirements.txt

In [ ]:
import sys, os
# Garante que a raiz do projeto esteja no path para 'import src...'
CWD = os.getcwd()
ROOT = CWD if os.path.isdir(os.path.join(CWD, 'src')) else os.path.dirname(CWD)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)  # paths relativos (models/, reports/) consistentes
print('Project root:', ROOT)

In [ ]:
from src.data_loader import load_raw_data, make_splits, make_tf_datasets, CLASS_NAMES
from src.models import build_cnn_a
from src.train import compile_and_train, save_history
from src.evaluate import plot_training_curves

## Dados

In [ ]:
images, labels = load_raw_data()
splits = make_splits(images, labels, seed=42)
train_ds, val_ds, test_ds = make_tf_datasets(splits, batch_size=64)

## Arquitetura

In [ ]:
model_a = build_cnn_a()
model_a.summary()

Note a camada `Flatten` seguida de `Dense(128)`: é responsável pela maior parte dos parâmetros da rede e, sem regularização, é onde o overfitting se instala.

## Treino
`EarlyStopping` (patience=5) monitora a `val_loss` e restaura os melhores pesos; `ModelCheckpoint` salva o melhor modelo por `val_accuracy`.

In [ ]:
history_a = compile_and_train(
    model_a, train_ds, val_ds,
    epochs=30, lr=1e-3,
    checkpoint_path='models/cnn_a_best.keras',
    patience=5, use_reduce_lr=False,
)
save_history(history_a, 'reports/cnn_a_history.json')

## Curvas de aprendizado

In [ ]:
plot_training_curves(history_a.history, title='CNN-A (Baseline)',
                     save_path='reports/figures/cnn_a_curves.png')

## Avaliação no conjunto de teste

In [ ]:
test_loss, test_acc = model_a.evaluate(test_ds, verbose=0)
print(f'CNN-A — acurácia no teste: {test_acc:.4f} | loss: {test_loss:.4f}')

**Leitura esperada.** A acurácia de treino tende a subir bem acima da de validação, e a `val_loss` para de cair (ou volta a subir) enquanto a `train_loss` continua caindo — a assinatura clássica de overfitting. Anote o valor de teste para comparar com a CNN-B.

*(Preencha os números reais após rodar.)*